# Basis-rotation grouping — fewer measurements for molecular expectation values

## The problem BRG addresses

Variational algorithms such as **VQE** estimate the ground-state energy of a
molecule by evaluating the expectation value $\langle\psi|H|\psi\rangle$ of an
electronic Hamiltonian $H$ on a trial state $|\psi\rangle$, over and over,
inside a classical optimization loop. That expectation value is what the
quantum computer has to measure — and measuring it is the expensive part.

A quantum computer reads its qubits in the computational ($Z$) basis only.
Written in qubit form, a molecular Hamiltonian is a sum of $O(N^4)$ Pauli terms
for $N$ spin orbitals, and most of them are *not* diagonal in that basis.
Measured naively that means one circuit per term, each needing many shots to
beat statistical noise. For molecules of practical size it is this measurement
cost — not the circuit depth — that makes VQE prohibitive on real hardware
(Gonthier *et al.*, Phys. Rev. Research **4**, 033154 (2022) puts numbers on
this, and compares the grouping schemes below).

**Grouping methods** attack exactly this. Terms that commute in a suitable
sense can be measured *simultaneously* from a single circuit, so the cost
becomes one circuit per group instead of one per term. Fewer groups means a
cheaper energy evaluation.

**Basis-rotation grouping** (BRG; Huggins *et al.*, npj Quantum Inf **7**, 23 (2021))
does the grouping on the *fermionic* Hamiltonian rather than on its Pauli
expansion. A double low-rank factorization of the two-electron integral tensor
rewrites $H$ as $L + 1$ groups, each a polynomial of number operators $n_p$ —
and therefore diagonal, once you rotate into that group's own orbital basis.
Because $L$ grows far more slowly than the $O(N^4)$ Pauli term count, the
number of circuits drops sharply. Measuring one group is
then simply: rotate the orbitals (an exact network of two-qubit Givens
rotations), then read every qubit.

This notebook factorizes a molecular Hamiltonian, shows how the circuit count
compares with Pauli grouping across a few molecules, estimates the
Hartree–Fock energy of H$_2$, and shows how the truncation threshold trades
circuits against accuracy.

> For a longer, gentler introduction to measurement optimization and where BRG
> sits among the alternatives, see PennyLane's
> [Measurement optimization](https://pennylane.ai/demos/tutorial_measurement_optimize)
> demo.

In [ ]:
import numpy as np
from pyscf import gto, scf

import qarp
from qarp.algorithms import BasisRotationAveraging
from qarp.blocks import ComputationalBasisStateBlock
from qarp.engines import QarpEngine
from qarp.operators import (
    JordanWigner,
    QubitWiseCommuting,
    basis_rotation_grouping,
)
from qarp.operators.integrals import restricted_integrals_to_fermion_operator
from qarp.operators.pyscf import integrals_from_mf, onv_from_mf


def converged_rhf(atom, basis="sto3g"):
    """A converged restricted Hartree-Fock object for a molecule."""
    mol = gto.M(atom=atom, basis=basis, verbose=0)
    mean_field = scf.RHF(mol)
    mean_field.kernel()
    return mean_field


# H2 / STO-3G at 0.735 A -- the running example throughout the notebook.
# `integrals_from_mf` is qarp's pyscf adapter: it returns the nuclear repulsion
# constant, the one-electron matrix and the two-electron tensor in the MO
# spatial basis, chemists' notation -- exactly what BRG consumes.
h2_mean_field = converged_rhf("H 0 0 0; H 0 0 0.735")
constant, one_electron, two_electron = integrals_from_mf(h2_mean_field)
n_qubits = 2 * one_electron.shape[0]
print(f"H2/STO-3G: {one_electron.shape[0]} spatial orbitals -> {n_qubits} qubits")

## Factorizing the one-electron and two-electron fermionic integrals

`basis_rotation_grouping` performs the factorization. Given the one-electron
matrix and the two-electron tensor, it returns the three components of the
grouping:

- `coefficients` — the per-group scalars $c_\ell$,
- `groups` — the per-group diagonal number-operator operators $G_\ell$,
- `rotations` — the per-group orbital rotation matrices $u_\ell$,

which together reconstruct the Hamiltonian exactly,

$$H = \text{const} + \sum_\ell c_\ell\, U(u_\ell)\, G_\ell\, U(u_\ell)^\dagger .$$

Group 0 is the corrected one-body term; the remaining groups come from the
two-electron double factorization, ordered by descending $|\lambda|$.

The factorization is *spin-summed* — it runs on the spatial $(pq|rs)$ tensor,
and the resulting spatial rotation is applied to both spin sublattices. That
keeps the number of groups at $N(N+1)/2 + 1$ for $N$ spatial orbitals.

Note that this function *only* builds the groups. It runs no circuit and
computes no expectation value — estimating $\langle H\rangle$ from these groups
is the job of the `BasisRotationAveraging` algorithm, two sections below.

In [ ]:
coefficients, groups, rotations = basis_rotation_grouping(one_electron, two_electron)

print(f"groups (L + 1):        {len(coefficients)}")
print(f"group 0 coefficient:   {coefficients[0]}  (the corrected one-body term)")
print(f"rotation matrix shape: {rotations[0].shape}  (one per group)")
print(f"group 1 operator:      {len(groups[1].terms)} number-operator terms")

## How many circuits does BRG save?

The group count *is* the circuit count for one energy evaluation, so it is the
number to compare. Below, for a few molecules: how many Pauli terms the qubit
Hamiltonian has, how many groups qubit-wise-commuting (QWC) Pauli grouping
needs, and how many BRG needs.

The gap widens quickly with system size — that is the difference between a
group count set by the orbital count and one set by the $O(N^4)$ Pauli
expansion.

In [ ]:
molecules = {
    "H2  (4 qubits)": "H 0 0 0; H 0 0 0.735",
    "H4  (8 qubits)": "H 0 0 0; H 0 0 1; H 0 0 2; H 0 0 3",
    "LiH (12 qubits)": "Li 0 0 0; H 0 0 1.6",
    "H2O (14 qubits)": "O 0 0 0; H 0 0.757 0.587; H 0 -0.757 0.587",
}

print(f"{'molecule':17}{'Pauli terms':>13}{'QWC groups':>13}{'BRG groups':>13}")
for label, atom in molecules.items():
    nuclear, h1, h2 = integrals_from_mf(converged_rhf(atom))

    # BRG: one group per retained factor, plus the one-body group.
    brg_groups = len(basis_rotation_grouping(h1, h2)[0])

    # QWC: one group per qubit-wise-commuting set of Pauli terms.
    operator = JordanWigner().encode_operator(
        restricted_integrals_to_fermion_operator(nuclear, h1, h2)
    )
    pauli_terms = [{q: p for q, p in term} for term in operator.terms if term]
    qwc_groups = QubitWiseCommuting().group(pauli_terms, 2 * h1.shape[0])

    print(f"{label:17}{len(pauli_terms):>13}{len(qwc_groups):>13}{brg_groups:>13}")

## Expectation value estimation

`BasisRotationAveraging` is a OpenQARP primitive algorithm. It takes a trial state
(`ket`) and the one-electron and two-electron tensors (`integrals`); the
nuclear repulsion constant (`constant`) is optional and defaults to `0.0`, but
is needed for a *total* molecular energy. The tensors and the constant assume
an electronic Hamiltonian in second-quantized form, and are exactly what
`qarp.operators.pyscf.integrals_from_mf` returned above.

From those inputs the algorithm builds one circuit per group, runs them on the
engine you choose, and combines the measurement counts into a single energy
estimate. Passing `n_shots=qarp.EXACT` makes it read exact Born probabilities
instead of sampling, which checks the method itself without shot noise.

In [ ]:
# 1. Create a trial state.  `onv_from_mf` gives the Hartree-Fock determinant
#    as an abab occupation-number vector (alpha and beta interleaved).
reference_onv = list(onv_from_mf(h2_mean_field))
hartree_fock = ComputationalBasisStateBlock(basis_state=reference_onv)

# 2. Build the BRG circuits from the integrals -- one circuit per group.
brg = BasisRotationAveraging(
    ket=hartree_fock, integrals=(one_electron, two_electron), constant=constant
)
brg.build()
print(f"circuits to run: {brg.n_groups}")

# 3. Select an engine and run them.  qarp.EXACT reads exact Born
#    probabilities, so the result carries no shot noise.
engine = QarpEngine(n_shots=qarp.EXACT)
engine.build([brg])
exact_energy = engine.run()[0]

# 4. Compare against the same expectation value computed *without* grouping:
#    the matrix element of the full, unfactorized Hamiltonian.
hamiltonian = restricted_integrals_to_fermion_operator(
    constant, one_electron, two_electron
)
state_index = sum(bit << qubit for qubit, bit in enumerate(reference_onv))
reference = hamiltonian.sparse_matrix(n_qubits).toarray()[state_index, state_index].real

print(f"BRG (exact readout): {exact_energy:.10f}")
print(f"dense <HF|H|HF>:     {reference:.10f}")
assert abs(exact_energy - reference) < 1e-6

In [ ]:
# The same estimate from finite sampling rather than exact probabilities:
# 8000 shots per group, on a seeded engine for reproducibility.
sampled = BasisRotationAveraging(
    ket=hartree_fock, integrals=(one_electron, two_electron), constant=constant
).build()
engine = QarpEngine(n_shots=8000, seed=42)
engine.build([sampled])
sampled_energy = engine.run()[0]

print(f"BRG (8000 shots):    {sampled_energy:.4f}")
print(f"dense <HF|H|HF>:     {reference:.4f}")
assert abs(sampled_energy - reference) < 0.05

## Discarding negligible Hamiltonian factors

`BasisRotationAveraging` accepts an optional `tolerance` parameter that tunes
the estimation error of the expectation value. A higher threshold discards more
negligible Hamiltonian factors, which lowers the number of groups — and so the
number of circuits to run — but increases the error. A lower threshold retains
more groups, which reduces the error.

The threshold is *relative*, measured against the largest factor
$|\lambda|_{\max}$, because the absolute scale of the factors varies from one
molecule and basis set to the next. The default `1e-8` keeps every factor that
is not numerically null.

The shift truncation introduces is *deterministic*: it comes from dropping part
of the Hamiltonian, not from statistical noise, so unlike shot noise it does
not average away as you add shots.

In [ ]:
for tolerance in (1e-8, 1e-2, 5e-2, 3e-1):
    probe = BasisRotationAveraging(
        ket=hartree_fock,
        integrals=(one_electron, two_electron),
        constant=constant,
        tolerance=tolerance,
    ).build()
    engine = QarpEngine(n_shots=qarp.EXACT)
    engine.build([probe])
    energy = engine.run()[0]
    print(
        f"tolerance={tolerance:>7.1e}: {probe.n_groups:>2} groups, "
        f"energy {energy:>14.10f}, error {abs(energy - reference):.2e}"
    )